# 05: docker-compose: orquestar varios servicios juntos


## Tu primer docker-compose.yml


Un docker-compose.yml es un archivo en formato YAML (un formato legible por humanos, como JSON pero más limpio) que describe tu infraestructura. Veamos uno simple con PostgreSQL:



**Archivo: docker-compose.yml**


```yml
# Archivo: docker-compose.yml
services:
    db:
        image: postgres:16
        environment:
            POSTGRES_USER: datos
            POSTGRES_PASSWORD: secreto123
            POSTGRES_DB: ventas
        ports:
            - "5432:5432"
```

```bash
# Levantar todos los servicios definidos en docker-compose.yml
docker compose up

# Levantar en background (detached)
docker compose up -d

# Ver qué servicios están corriendo
docker compose ps

# Parar todos los servicios
docker compose down

# Parar y BORRAR los volúmenes (borra datos)
docker compose down -v

```

## Múltiples servicios: la potencia real


La verdadera magia de docker-compose llega cuando tienes VARIOS servicios que necesitan hablar entre sí.

 Veamos un ejemplo con PostgreSQL + Redis + una app Python:


```yml
# Archivo: docker-compose.yml | Multi-Services

services:
  db: 
    image: postgres:16
    environment:
      POSTGRES_USER: datos
      POSTGRES_PASSWORD: secreto123
      POSTGRES_DB: ventas
    ports:
      - "5432:5432"

  cache:
    image: redis:7-alpine
    ports:
    - "6379:6379"
  
  app:
    build: .
    environment:
      DATABASE_URL: postgressql://datos:secreto123@db:5432/analytics
      REDIS_URL: redis://cache:6379
    depends_on:
      - db
      - cache
    ports:
      - "8000:8000"
```

## depends_on: el orden importa



Observa la clave depends_on en el servicio "app". 

- Le dice a Docker Compose: "no arranques la app hasta que db y cache estén levantados". 
- Es como decir "no empieces a cocinar la paella hasta que el fuego esté encendido". 
- Sin depends_on, los servicios arrancan en cualquier orden y tu app podría intentar conectarse a PostgreSQL antes de que esté listo.


1. depends_on solo garantiza que el CONTENEDOR ha arrancado, no que el SERVICIO dentro esté listo.
2.  PostgreSQL tarda unos segundos en inicializarse después de que su contenedor arranque. 
3.  Para producción se usan healthchecks, pero para desarrollo local depends_on suele ser suficiente.



```yml
# Healtcheck: Compose espera a que el servicio esté DE VERDAD listo
services:
    postgres:
        image: postgres:16
        environment:
            POSTGRES_PASSWORD: secreto123
        healtcheck: 
        test: ["CMD-SHELL", "pg_isready - U postgres"]
        interval: 3s
        timeout: 3s
        retries: 5

    etl:
        build: .
        depends_on:
            postgres:
                condition: service_healthy # <- esperar al healthcheck, no solo al arranque
```

## La magia del DNS interno: los servicios se encuentran por nombre


1. Fíjate en DATABASE_URL: postgresql://datos:secreto123@db:5432/analytics. 
    - El "db" en esa URL NO es localhost — es el NOMBRE del servicio definido en docker-compose.yml. 
    - Docker Compose crea una red interna donde cada servicio puede encontrar a los demás por su nombre. 
    - Es como si en un edificio de oficinas cada empresa tuviera un buzón con su nombre en el hall — no necesitas saber el número de puerta, solo el nombre.


## build vs image: usar tu propio Dockerfile



Has visto que "db" y "cache" usan image: (imágenes pre-hechas de Docker Hub), pero "app" usa build: . (construye una imagen desde tu Dockerfile local). Puedes mezclar ambos estilos libremente:

- **image:** postgres:16 → "Usa la imagen postgres:16 de Docker Hub tal cual"
- **build:** . → "Construye una imagen usando el Dockerfile que está en el directorio actual"
- **build:** ./mi-servicio → "Construye usando el Dockerfile dentro de la carpeta mi-servicio"


## Comandos esenciales de docker-compose


```bash
# Levantar servicios en background
docker compose up -d

# Ver logs de todos los servicios
docker compose logs

# Verlos de UN servicio específico en tiempo real
docker compose logs -f db

# Parar servicios sin borrarlos
docker compose stop

# Parar y borrar contenedores + redes (datos en volúmenes se mantienen)
docker compose down

# Reconstruir imágenes (cuando cambias tu Dockerfile o código)
docker compose up -d --build

# Ejecutar un comando dentro de un servicio que ya está corriendo
docker compose exec db psql -U datos -d analytics

# Escalar un servicio (crear múltiples instancias) en tu Docker real
docker compose up -d --scale app=3
```

/